# Tuần 7 - Ngày 2

# Dự án cuối khóa "THE PRICE IS RIGHT" (Đoán giá sản phẩm)

Tuần này chúng ta sẽ fine-tune (tinh chỉnh) một mô hình mã nguồn mở!

Đây là một mô hình có thể ước tính giá của một sản phẩm dựa trên mô tả của nó.

# Lịch trình các ngày

NGÀY 1: QLoRA (kỹ thuật fine-tune tiết kiệm bộ nhớ)  
NGÀY 2: Chuẩn bị dữ liệu prompt và mô hình gốc (Base Model)  
NGÀY 3: Huấn luyện (Train) - Phần 1  
NGÀY 4: Huấn luyện (Train) - Phần 2  
NGÀY 5: Đánh giá (Eval)  

## Đầu tiên, chúng ta cần tải lên bộ dữ liệu cuối cùng


In [1]:
# Nạp thư viện: biến môi trường, đăng nhập HuggingFace, lớp Item tuỳ biến, tokenizer và vẽ biểu đồ
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.items import Item
from tqdm.notebook import tqdm
from transformers import AutoTokenizer
import matplotlib.pyplot as plt


In [ ]:
LITE_MODE = False  # True: dùng bộ dữ liệu rút gọn (lite) để chạy thử nhanh

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)  # Đăng nhập HuggingFace Hub để có quyền tải/đẩy dữ liệu


In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

# Tải bộ dữ liệu train/val/test đã xử lý sẵn từ HuggingFace Hub
train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")


In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-3B"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)  # Tokenizer của mô hình gốc, dùng để đếm/token hóa dữ liệu


In [ ]:
# Đếm số token trong phần summary của từng item để khảo sát độ dài dữ liệu
token_counts = [item.count_tokens(tokenizer) for item in tqdm(items)]


In [ ]:
# Vẽ histogram phân bố số token của phần summary
plt.figure(figsize=(15, 6))
plt.title(f"Tokens in Summary: Avg {sum(token_counts)/len(token_counts):,.1f} and highest {max(token_counts):,}\n")
plt.xlabel('Number of tokens in summary')
plt.ylabel('Count')
plt.hist(token_counts, rwidth=0.7, color="skyblue", bins=range(0, 200, 10))
plt.show()


In [ ]:
CUTOFF = 110  # Ngưỡng số token tối đa cho prompt, dùng để cắt bớt các item quá dài
cut = len([count for count in token_counts if count > CUTOFF])
print(f"With this CUTOFF, we will truncate {cut:,} items which is {cut/len(items):.1%}")


In [ ]:
print(train[0].summary)  # Xem thử nội dung summary của item đầu tiên


In [ ]:
# Tạo prompt (đầu vào) và completion (đầu ra mong muốn) cho từng item, cắt theo CUTOFF token
for item in tqdm(train+val):
    item.make_prompts(tokenizer, CUTOFF, True)
for item in tqdm(test):
    item.make_prompts(tokenizer, CUTOFF, False)


In [ ]:
# Kiểm tra nhanh prompt/completion vừa tạo cho item test đầu tiên
print("PROMPT:")
print(test[0].prompt)
print("COMPLETION:")
print(test[0].completion)


In [ ]:
# Đếm số token của toàn bộ prompt + completion để kiểm tra độ dài đầu vào thực tế cho mô hình
prompt_token_counts = [item.count_prompt_tokens(tokenizer) for item in tqdm(items)]


In [ ]:
# Vẽ histogram phân bố số token của prompt + completion
plt.figure(figsize=(15, 6))
plt.title(f"Tokens: Avg {sum(prompt_token_counts)/len(prompt_token_counts):,.1f} and highest {max(prompt_token_counts):,}\n")
plt.xlabel('Number of tokens in prompt and the completion')
plt.ylabel('Count')
plt.hist(prompt_token_counts, rwidth=0.7, color="gold", bins=range(0, 200, 10))
plt.show()


In [ ]:
username = "ed-donner"
dataset = f"{username}/items_prompts_lite" if LITE_MODE else f"{username}/items_prompts_full"

# Đẩy bộ prompt/completion đã tạo lên HuggingFace Hub để dùng cho bước huấn luyện
Item.push_prompts_to_hub(dataset, train, val, test)


Đây là các bộ dữ liệu (dataset) trên HuggingFace:

https://huggingface.co/datasets/ed-donner/items_prompts_lite

https://huggingface.co/datasets/ed-donner/items_prompts_full

Vui lòng xem notebook này trên Google Colab:

https://colab.research.google.com/drive/1wO3lNMrMfprlJZF4X9fSsQ8tYC3SRZbh?usp=sharing


## Ghi note giải thích notebook

### Tóm tắt quy trình của notebook

Notebook Ngày 2 chuẩn bị dữ liệu để fine-tune mô hình đoán giá sản phẩm. Quy trình gồm: (1) đăng nhập HuggingFace Hub, (2) tải bộ dữ liệu item đã xử lý (train/val/test), (3) tải tokenizer của mô hình gốc Llama-3.2-3B, (4) khảo sát độ dài (số token) của phần summary để chọn ngưỡng cắt (CUTOFF) hợp lý, (5) tạo prompt và completion cho từng item dựa trên CUTOFF đó, (6) kiểm tra lại độ dài prompt sau khi tạo, và cuối cùng (7) đẩy bộ dữ liệu prompt hoàn chỉnh lên HuggingFace Hub để dùng cho bước huấn luyện ở Ngày 3-4.

### Ý nghĩa chính của notebook

Trước khi huấn luyện, mô hình ngôn ngữ cần dữ liệu ở đúng định dạng: mỗi item phải có một "prompt" (mô tả sản phẩm) và một "completion" (giá đúng của sản phẩm) đã được token hóa với độ dài phù hợp. Việc khảo sát số token qua biểu đồ histogram giúp chọn ngưỡng CUTOFF cân bằng giữa việc giữ đủ thông tin và tránh dữ liệu quá dài gây tốn tài nguyên khi huấn luyện. Sau bước này, dữ liệu đã sẵn sàng ở dạng chuẩn để nạp vào quá trình fine-tune QLoRA.

### Giải thích từng cell

**Cell 1 (markdown - giới thiệu):** Nêu lại mục tiêu dự án cả tuần và cho biết Ngày 2 sẽ bắt đầu bằng việc tải lên bộ dữ liệu cuối cùng.

**Cell 2 (import thư viện):** Nạp các thư viện cần cho việc đăng nhập HuggingFace, xử lý dữ liệu (`Item`), token hóa (`AutoTokenizer`) và vẽ biểu đồ (`matplotlib`). Cần thiết vì các cell sau đều phụ thuộc vào các thư viện này.

**Cell 3 (đăng nhập HuggingFace):** Đặt cờ `LITE_MODE`, nạp biến môi trường chứa `HF_TOKEN` và đăng nhập vào HuggingFace Hub. Cần thiết để có quyền tải và đẩy (push) dữ liệu lên Hub ở các bước sau.

**Cell 4 (tải dữ liệu):** Tải 3 tập train/val/test từ HuggingFace Hub bằng `Item.from_hub`, rồi gộp lại thành `items`. Kết quả này là đầu vào cho toàn bộ các bước khảo sát và tạo prompt phía sau.

**Cell 5 (tải tokenizer):** Tải tokenizer tương ứng với mô hình gốc `Llama-3.2-3B`. Tokenizer này được dùng xuyên suốt để đếm và mã hóa văn bản thành token.

**Cell 6 (đếm token summary):** Đếm số token trong phần summary của từng item, kết quả dùng để vẽ biểu đồ ở cell tiếp theo.

**Cell 7 (vẽ histogram summary):** Trực quan hóa phân bố số token để người dùng thấy phần lớn summary dài bao nhiêu token, từ đó hỗ trợ chọn CUTOFF.

**Cell 8 (chọn CUTOFF):** Đặt ngưỡng `CUTOFF = 110` token và tính xem có bao nhiêu item sẽ bị cắt bớt (truncate) nếu áp dụng ngưỡng này, giúp đánh giá mức độ ảnh hưởng của việc cắt dữ liệu.

**Cell 9 (xem thử summary):** In summary của item đầu tiên để kiểm tra định dạng dữ liệu thực tế trước khi tạo prompt.

**Cell 10 (tạo prompt/completion):** Gọi `item.make_prompts` cho từng item trong train/val (kèm nhãn) và test (không kèm nhãn), áp dụng CUTOFF vừa chọn. Đây là bước quan trọng nhất, biến dữ liệu thô thành định dạng huấn luyện.

**Cell 11 (kiểm tra prompt/completion):** In thử prompt và completion của item test đầu tiên để xác nhận định dạng đúng như mong đợi.

**Cell 12 (đếm token của prompt):** Đếm tổng số token của cả prompt lẫn completion cho từng item, dùng để vẽ biểu đồ kiểm tra độ dài thực tế sẽ đưa vào mô hình.

**Cell 13 (vẽ histogram prompt):** Trực quan hóa phân bố số token của prompt + completion, giúp xác nhận rằng CUTOFF đã chọn cho ra độ dài dữ liệu hợp lý.

**Cell 14 (đẩy dữ liệu lên Hub):** Gọi `Item.push_prompts_to_hub` để lưu bộ dữ liệu prompt/completion đã hoàn thiện lên HuggingFace Hub, sẵn sàng cho bước huấn luyện.

**Cell 15 (markdown - liên kết):** Cung cấp link tới 2 bộ dataset vừa tạo trên HuggingFace và link notebook Google Colab để tiếp tục thực hành (do cần GPU).

> Notebook này dùng để biến dữ liệu sản phẩm thô thành các cặp prompt/completion đã token hóa đúng chuẩn, vì vậy nó cần đăng nhập Hub, tải dữ liệu, khảo sát độ dài bằng tokenizer, rồi tạo và lưu lại prompt để mô hình có thể học từ đó ở bước huấn luyện tiếp theo.

### Mục tiêu cuối cùng

Sau notebook này, ta có một bộ dữ liệu prompt/completion hoàn chỉnh, đã được token hóa và cắt theo ngưỡng hợp lý, được lưu trên HuggingFace Hub - sẵn sàng để nạp vào quá trình fine-tune QLoRA ở Ngày 3 và Ngày 4.
